# Connect 4 Policy Analysis

This notebook compares the available agents without using `tournament.py`, so the bracket BYE issue and the winner-accounting issue do not affect the results.

Agents included:

- `random`: baseline random policy
- `v0`: `groups/my-solution-v0` (no heuristic version)
- `v1`: `groups/my-solution-v1` (heuristic version)
- `current`: `groups/my-solution` (heuristic + transposition-table agent)

The Mojo agents now accept `depth_obj` at runtime, so the same imported policy can be benchmarked at different depths by setting `policy.search_depth`.

In [ ]:
from __future__ import annotations

import importlib
import itertools
import os
import re
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
assert (ROOT / "connect4" / "connect_state.py").exists(), ROOT

from connect4.connect_state import ConnectState

## Helpers

These helpers play honest matches: the first policy is always Red (`-1`) and the second policy is Yellow (`+1`). For fair comparisons, `match()` alternates who starts.

In [ ]:
@dataclass(frozen=True)
class PolicySpec:
    name: str
    module: str
    cls: str
    source_dir: str | None = None
    mojo_file: str | None = None


POLICIES = {
    "random": PolicySpec("random", "groups.random-group.policy",   "RandomPolicy",            None,                   None),
    "v0":     PolicySpec("v0",     "groups.my-solution-v0.policy", "OhYes",                   "groups/my-solution-v0", "solution_v0.mojo"),
    "v1":     PolicySpec("v1",     "groups.my-solution-v1.policy", "NegamaxHeuristics",        "groups/my-solution-v1", "solution_v1.mojo"),
    "current":PolicySpec("current","groups.my-solution.policy",    "NegamaxAdaptativeDeepening","groups/my-solution",   "solution.mojo"),
}
SEARCH_DEPTH = 8


def load_policy_class(spec: PolicySpec):
    module = importlib.import_module(spec.module)
    return getattr(module, spec.cls)


def make_policy(spec: PolicySpec, depth: int | None = None):
    cls = load_policy_class(spec)
    policy = cls()
    if hasattr(policy, "mount"):
        policy.mount()
    if depth is not None and hasattr(policy, "search_depth"):
        policy.search_depth = depth
    return policy


def play_game(first_spec: PolicySpec, second_spec: PolicySpec, depth: int | None = None, max_moves: int = 42):
    first = make_policy(first_spec, depth)
    second = make_policy(second_spec, depth)
    state = ConnectState()
    moves = []

    while not state.is_final() and len(moves) < max_moves:
        policy = first if state.player == -1 else second
        action = int(policy.act(state.board))
        moves.append(action)
        state = state.transition(action)

    winner_color = state.get_winner()
    if winner_color == -1:
        winner = first_spec.name
    elif winner_color == 1:
        winner = second_spec.name
    else:
        winner = "draw"
    return {"first": first_spec.name, "second": second_spec.name, "winner": winner, "moves": len(moves), "sequence": moves}


def match(a: PolicySpec, b: PolicySpec, games: int = 10, depth: int | None = None):
    rows = []
    for i in range(games):
        if i % 2 == 0:
            row = play_game(a, b, depth=depth)
        else:
            row = play_game(b, a, depth=depth)
        row["game"] = i + 1
        row["pair"] = f"{a.name} vs {b.name}"
        rows.append(row)
    return pd.DataFrame(rows)


def summarize_match(df: pd.DataFrame, a: str, b: str):
    counts = df["winner"].value_counts().to_dict()
    total = len(df)
    return {
        "pair": f"{a} vs {b}",
        "a": a,
        "b": b,
        "games": total,
        "a_wins": counts.get(a, 0),
        "b_wins": counts.get(b, 0),
        "draws": counts.get("draw", 0),
        "a_win_rate": counts.get(a, 0) / total,
        "b_win_rate": counts.get(b, 0) / total,
        "avg_moves": df["moves"].mean(),
    }

## Current Repository Comparison

This section compares the versions at their policy default depths. The detected `comptime DEPTH` values are shown for reference, but normal calls now pass `search_depth` at runtime.

In [ ]:
depth_constants = []
for name, spec in POLICIES.items():
    if spec.source_dir and spec.mojo_file:
        text = (ROOT / spec.source_dir / spec.mojo_file).read_text()
        m = re.search(r"comptime\s+DEPTH\s*:\s*Int\s*=\s*(\d+)", text)
        depth_constants.append({"agent": name, "comptime_depth": int(m.group(1)) if m else None})
    else:
        depth_constants.append({"agent": name, "comptime_depth": None})

pd.DataFrame(depth_constants)

In [ ]:
GAMES_PER_PAIR = 10

pair_results = []
game_logs = []

for a_name, b_name in itertools.combinations(POLICIES.keys(), 2):
    a = POLICIES[a_name]
    b = POLICIES[b_name]
    df = match(a, b, games=GAMES_PER_PAIR, depth=SEARCH_DEPTH)
    game_logs.append(df)
    pair_results.append(summarize_match(df, a_name, b_name))

repo_games = pd.concat(game_logs, ignore_index=True)
repo_summary = pd.DataFrame(pair_results)
repo_summary

In [ ]:
plot_df = repo_summary.melt(
    id_vars=["pair"],
    value_vars=["a_win_rate", "b_win_rate"],
    var_name="side",
    value_name="win_rate",
)
plot_df["agent"] = plot_df.apply(lambda r: repo_summary.loc[repo_summary["pair"] == r["pair"], "a"].iloc[0] if r["side"] == "a_win_rate" else repo_summary.loc[repo_summary["pair"] == r["pair"], "b"].iloc[0], axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
for pair, sub in plot_df.groupby("pair"):
    ax.bar([f"{pair}\n{sub.iloc[i]['agent']}" for i in range(len(sub))], sub["win_rate"])
ax.set_ylim(0, 1)
ax.set_ylabel("Win rate")
ax.set_title(f"Pairwise win rates at depth {SEARCH_DEPTH}, {GAMES_PER_PAIR} alternating-start games per pair")
ax.tick_params(axis="x", rotation=70)
plt.tight_layout()

## Win Rate Against Random

This isolates the baseline question: how reliably does each non-random agent beat random play?

In [ ]:
random_rows = []
for name in ["v0", "v1", "current"]:
    df = match(POLICIES[name], POLICIES["random"], games=20, depth=SEARCH_DEPTH)
    random_rows.append(summarize_match(df, name, "random"))

random_summary = pd.DataFrame(random_rows)
random_summary[["a", "games", "a_wins", "draws", "a_win_rate", "avg_moves"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(random_summary["a"], random_summary["a_win_rate"], color=["#6b7280", "#2563eb", "#059669"])
ax.set_ylim(0, 1)
ax.set_ylabel("Win rate vs random")
ax.set_title(f"Baseline win rate against random at depth {SEARCH_DEPTH}")
plt.tight_layout()

## Move Latency At Depth 8

This measures `act()` runtime on sampled positions. It is a practical performance proxy: lower latency means the agent can search more within a fixed time budget.

In [ ]:
def collect_positions(first: PolicySpec, second: PolicySpec, games: int = 4, depth: int | None = None):
    positions = []
    for i in range(games):
        a, b = (first, second) if i % 2 == 0 else (second, first)
        p1 = make_policy(a, depth=depth)
        p2 = make_policy(b, depth=depth)
        state = ConnectState()
        while not state.is_final():
            positions.append(state.board.copy())
            policy = p1 if state.player == -1 else p2
            state = state.transition(int(policy.act(state.board)))
    return positions


def latency_benchmark(spec: PolicySpec, positions: list[np.ndarray], repeats: int = 3, depth: int | None = None):
    policy = make_policy(spec, depth=depth)
    samples = []
    for board in positions:
        for _ in range(repeats):
            t0 = time.perf_counter()
            _ = int(policy.act(board))
            samples.append(time.perf_counter() - t0)
    arr = np.array(samples)
    return {
        "agent": spec.name,
        "samples": len(samples),
        "mean_ms": arr.mean() * 1000,
        "median_ms": np.median(arr) * 1000,
        "p95_ms": np.quantile(arr, 0.95) * 1000,
    }


positions = collect_positions(POLICIES["v1"], POLICIES["current"], games=4, depth=SEARCH_DEPTH)
latency_df = pd.DataFrame([latency_benchmark(POLICIES[name], positions, depth=SEARCH_DEPTH) for name in ["v0", "v1", "current"]])
latency_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(latency_df["agent"], latency_df["median_ms"], color=["#6b7280", "#2563eb", "#059669"])
ax.set_ylabel("Median act() latency (ms)")
ax.set_title(f"Policy decision latency at depth {SEARCH_DEPTH} on shared sampled positions")
plt.tight_layout()

## Depth Sweep: v1 vs Current

This section benchmarks `v1` and `current` by setting `policy.search_depth` directly before each timing run.

Use fewer depths or fewer sampled positions if high-depth runs are slow.

In [ ]:
def runtime_depth_latency(agent_name: str, depth: int, positions: list[np.ndarray], repeats: int = 1):
    spec = POLICIES[agent_name]
    policy = make_policy(spec, depth=depth)
    samples = []
    for board in positions:
        for _ in range(repeats):
            t0 = time.perf_counter()
            _ = int(policy.act(board))
            samples.append(time.perf_counter() - t0)
    arr = np.array(samples)
    return {
        "agent": agent_name,
        "depth": depth,
        "samples": len(samples),
        "mean_ms": arr.mean() * 1000,
        "median_ms": np.median(arr) * 1000,
        "p95_ms": np.quantile(arr, 0.95) * 1000,
    }

In [ ]:
depth_ranges = {
    "v1":      [2, 4, 6, 8, 10, 12],
    "current": [2, 4, 6, 8, 10, 12, 14, 16],
}
depth_rows = []

for agent_name, depths in depth_ranges.items():
    for depth in depths:
        print(f"benchmarking {agent_name} depth={depth}...")
        depth_rows.append(runtime_depth_latency(agent_name, depth, positions[:12], repeats=1))

depth_latency_df = pd.DataFrame(depth_rows)
depth_latency_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for agent_name, sub in depth_latency_df.groupby("agent"):
    sub = sub.sort_values("depth")
    ax.plot(sub["depth"], sub["median_ms"], marker="o", label=agent_name)
ax.set_xlabel("Runtime search depth")
ax.set_ylabel("Median act() latency (ms)")
ax.set_title("Depth vs performance: heuristic v1 vs current TT agent")
ax.legend()
plt.tight_layout()

## Report Additions: Version Comparison, Latency vs Depth, Decision Time by Move

These cells generate the figures used in the PDF report.
Figures are saved to `document/figures/` alongside the typst source.

**Corrected POLICIES dict** — class names match the actual policy files (v0 → `OhYes`, v1 → `NegamaxHeuristics`, v2 → `NegamaxTranspositionTable`, current → `NegamaxAdaptativeDeepening`).

In [ ]:
from pathlib import Path

POLICIES_REPORT = {
    "random":  PolicySpec("random",  "groups.random-group.policy",   "RandomPolicy"),
    "v0":      PolicySpec("v0",      "groups.my-solution-v0.policy", "OhYes"),
    "v1":      PolicySpec("v1",      "groups.my-solution-v1.policy", "NegamaxHeuristics"),
    "v2":      PolicySpec("v2",      "groups.my-solution-v2.policy", "NegamaxTranspositionTable"),
    "current": PolicySpec("current", "groups.my-solution.policy",    "NegamaxAdaptativeDeepening"),
}

FIGURE_DIR = Path("document/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

STYLE = {
    "v0":      {"color": "#6b7280", "label": "v0\n(no heuristic)"},
    "v1":      {"color": "#2563eb", "label": "v1\n(heuristic)"},
    "v2":      {"color": "#d97706", "label": "v2\n(heuristic+TT)"},
    "current": {"color": "#059669", "label": "current\n(TT+adapt.)"},
}

REPORT_DEPTH = 8
print("Policies loaded:", list(POLICIES_REPORT))

### Latency vs Depth: v1 (no TT) vs v2 (TT) vs current (TT + adaptive)

The transposition table avoids re-evaluating positions seen earlier in the search tree.
This cell measures per-move decision time at depths 2–8 for all three versions,
showing both the exponential cost of depth and the TT speedup.

In [ ]:
# Fewer samples at deep depths to keep runtime manageable
N_POS_BY_DEPTH = {2: 20, 4: 20, 6: 15, 8: 15, 10: 8, 12: 5, 14: 3, 16: 2}
DEPTH_RANGES = {
    "v1":      [2, 4, 6, 8, 10, 12],
    "v2":      [2, 4, 6, 8, 10, 12, 14, 16],
    "current": [2, 4, 6, 8, 10, 12, 14, 16],
}
# Adaptive deepening budget: timeout=10s → budget=9s → agent stops when elapsed×7.5≥9s
ADAPTIVE_BUDGET_MS = 9000 / 7.5   # 1200 ms threshold

latency_report_rows = []
for agent_name, depths in DEPTH_RANGES.items():
    spec = POLICIES_REPORT[agent_name]
    for d in depths:
        n = N_POS_BY_DEPTH.get(d, 3)
        print(f"  {agent_name} depth={d} (N={n})...", flush=True)
        policy = make_policy(spec, d)
        state  = ConnectState()
        samples = []
        for _ in range(n):
            if state.is_final():
                state = ConnectState()
            t0 = time.perf_counter()
            action = int(policy.act(state.board))
            samples.append((time.perf_counter() - t0) * 1000)
            state = state.transition(action)
        arr = np.array(samples)
        latency_report_rows.append({
            "agent": agent_name, "depth": d,
            "median_ms": float(np.median(arr)),
            "p25_ms":    float(np.quantile(arr, 0.25)),
            "p75_ms":    float(np.quantile(arr, 0.75)),
        })

latency_report_df = pd.DataFrame(latency_report_rows)
latency_report_df

In [ ]:
agent_style = {
    "v1":      {"color": STYLE["v1"]["color"],      "label": "v1 (no TT)",             "ls": "--", "marker": "s"},
    "v2":      {"color": STYLE["v2"]["color"],      "label": "v2 (TT, static depth)",  "ls": "-.", "marker": "D"},
    "current": {"color": STYLE["current"]["color"], "label": "current (TT + adaptive)","ls": "-",  "marker": "o"},
}

fig, ax = plt.subplots(figsize=(7, 4.5))
all_depths = sorted({d for ds in DEPTH_RANGES.values() for d in ds})
for agent_name, sub in latency_report_df.groupby("agent"):
    sub = sub.sort_values("depth")
    s   = agent_style[agent_name]
    ax.plot(sub["depth"], sub["median_ms"], marker=s["marker"], color=s["color"],
            linewidth=2, markersize=7, label=s["label"], linestyle=s["ls"])
    ax.fill_between(sub["depth"], sub["p25_ms"], sub["p75_ms"],
                    alpha=0.12, color=s["color"])

ax.axhline(ADAPTIVE_BUDGET_MS, color="crimson", linestyle=":", linewidth=1.5,
           label=f"Adaptive budget ({ADAPTIVE_BUDGET_MS:.0f} ms, timeout=10 s)")
ax.annotate("adaptive deepening\nstops here →",
            xy=(14, ADAPTIVE_BUDGET_MS), xytext=(10, ADAPTIVE_BUDGET_MS * 4.5),
            arrowprops=dict(arrowstyle="->", color="crimson"),
            color="crimson", fontsize=8)
ax.set_xlabel("Search depth")
ax.set_ylabel("Median decision time (ms)")
ax.set_yscale("log")
ax.set_xticks(all_depths)
ax.set_title("Decision time vs search depth  (v1 → depth 12, v2/current → depth 16)\n"
             "shaded band = IQR, log scale")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig_latency_depth.pdf", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "fig_latency_depth.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_latency_depth")

### Version Comparison: v1 vs v2 vs current vs Random

Compares win rate of each version against the random baseline at depth=8 (N=30, alternating starts).
Error bars show ±1 binomial standard deviation (√(p(1−p)/n)).

In [ ]:
import math

VERSION_GAMES = 30

version_rows = []
for name in ("v1", "v2", "current"):
    spec = POLICIES_REPORT[name]
    rnd  = POLICIES_REPORT["random"]
    print(f"  {name} vs random ({VERSION_GAMES} games, depth={REPORT_DEPTH})...", flush=True)
    df = match(spec, rnd, games=VERSION_GAMES, depth=REPORT_DEPTH)
    wins = int((df["winner"] == name).sum())
    p = wins / VERSION_GAMES
    std_err = math.sqrt(p * (1 - p) / VERSION_GAMES) if 0 < p < 1 else 0.0
    version_rows.append({"version": name, "wins": wins, "n": VERSION_GAMES,
                          "win_rate": p, "std_err": std_err})

version_report_df = pd.DataFrame(version_rows)
version_report_df

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.5))
versions   = version_report_df["version"].tolist()
x          = np.arange(len(versions))
bar_colors = [STYLE[v]["color"] for v in versions]

ax.bar(x, version_report_df["win_rate"], color=bar_colors, width=0.5, alpha=0.85)
for xi, (_, row) in zip(x, version_report_df.iterrows()):
    ax.errorbar(xi, row["win_rate"], yerr=row["std_err"],
                fmt="none", color="black", capsize=6, linewidth=1.5)
    ax.text(xi, min(row["win_rate"] + row["std_err"] + 0.04, 1.1),
            f"{row['wins']}/{row['n']}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([STYLE[v]["label"] for v in versions])
ax.set_ylim(0, 1.18)
ax.set_ylabel("Win rate vs random")
ax.set_title(f"Version comparison (depth={REPORT_DEPTH}, N={VERSION_GAMES} games)\n±1 binomial std error")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig_version_comparison.pdf", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "fig_version_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_version_comparison")

### Color Analysis: Win Rate as First vs Second Player (v1 vs v2 vs current)

Checks whether performance differs by player color against the random baseline.
Error bars show ±1 binomial std error.

In [ ]:
N_COLOR = 15

color_rows = []
for name in ("v1", "v2", "current"):
    spec = POLICIES_REPORT[name]
    rnd  = POLICIES_REPORT["random"]
    print(f"  color {name}...", flush=True)

    p1_games = [play_game(spec, rnd, depth=REPORT_DEPTH) for _ in range(N_COLOR)]
    p1_wins  = sum(g["winner"] == name for g in p1_games)
    p1_wr    = p1_wins / N_COLOR
    p1_err   = math.sqrt(p1_wr * (1 - p1_wr) / N_COLOR) if 0 < p1_wr < 1 else 0.0

    p2_games = [play_game(rnd, spec, depth=REPORT_DEPTH) for _ in range(N_COLOR)]
    p2_wins  = sum(g["winner"] == name for g in p2_games)
    p2_wr    = p2_wins / N_COLOR
    p2_err   = math.sqrt(p2_wr * (1 - p2_wr) / N_COLOR) if 0 < p2_wr < 1 else 0.0

    color_rows += [
        {"version": name, "role": "First (P1)",  "win_rate": p1_wr, "std_err": p1_err, "wins": p1_wins},
        {"version": name, "role": "Second (P2)", "win_rate": p2_wr, "std_err": p2_err, "wins": p2_wins},
    ]

color_df = pd.DataFrame(color_rows)
color_df

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.5))
versions   = ["v1", "v2", "current"]
roles      = ["First (P1)", "Second (P2)"]
role_color = {"First (P1)": "#1d4ed8", "Second (P2)": "#b45309"}
x = np.arange(len(versions))
w = 0.35

for j, role in enumerate(roles):
    sub = color_df[color_df["role"] == role].set_index("version").reindex(versions)
    off = (j - 0.5) * w
    ax.bar(x + off, sub["win_rate"], w, label=role,
           color=role_color[role], alpha=0.82)
    for i, (_, row) in enumerate(sub.iterrows()):
        ax.errorbar(x[i] + off, row["win_rate"], yerr=row["std_err"],
                    fmt="none", color="black", capsize=4, linewidth=1.2)

ax.set_xticks(x)
ax.set_xticklabels([STYLE[v]["label"] for v in versions])
ax.set_ylim(0, 1.15)
ax.set_ylabel(f"Win rate vs random (N={N_COLOR} per color)")
ax.set_title("Win rate by player color: v1 vs v2 vs current\n(±1 binomial std error)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig_color_analysis.pdf", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "fig_color_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_color_analysis")

### Decision Time by Move Number: v1 vs v2 vs current

Tracks how per-move decision time evolves as a game progresses (measured by pieces already on the board).
In the opening (few pieces) the TT is cold; by the mid-game it accumulates hits, reducing redundant search.
This shows where the adaptive deepening mechanism has the most room to exploit time savings.

In [ ]:
N_TIMING_GAMES = 20   # more games → smoother curves and higher piece-count coverage

def make_policy_with_timeout(spec, depth=None, timeout=None):
    """Like make_policy but passes optional timeout to mount() for adaptive deepening."""
    cls = load_policy_class(spec)
    p   = cls()
    if hasattr(p, "mount"):
        p.mount(timeout=timeout)
    if depth is not None and hasattr(p, "search_depth"):
        p.search_depth = depth
    return p


def time_by_pieces(spec, opp_spec, n_games=20, depth=8):
    """Record spec's decision time vs pieces-on-board (spec always plays as P1/Red)."""
    records = []
    for _ in range(n_games):
        pol   = make_policy(spec,     depth)
        opp   = make_policy(opp_spec, depth)
        state = ConnectState()
        while not state.is_final():
            n_pieces = int(np.sum(state.board != 0))
            if state.player == -1:   # spec's turn (P1 = Red = -1)
                t0     = time.perf_counter()
                action = int(pol.act(state.board))
                records.append({"pieces": n_pieces,
                                 "ms": (time.perf_counter() - t0) * 1000,
                                 "agent": spec.name})
            else:
                action = int(opp.act(state.board))
            state = state.transition(action)
    return pd.DataFrame(records)


timing_frames = []
rnd = POLICIES_REPORT["random"]
for agent_name in ("v0", "v1", "v2", "current"):
    print(f"  timing {agent_name}...", flush=True)
    timing_frames.append(
        time_by_pieces(POLICIES_REPORT[agent_name], rnd,
                       n_games=N_TIMING_GAMES, depth=REPORT_DEPTH))

timing_df = pd.concat(timing_frames, ignore_index=True)
timing_df["piece_bin"] = (timing_df["pieces"] // 4) * 4
timing_summary = (timing_df.groupby(["agent", "piece_bin"])["ms"]
                  .agg(median="median",
                       q25=lambda x: x.quantile(0.25),
                       q75=lambda x: x.quantile(0.75))
                  .reset_index())
timing_summary

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
agent_style2 = {
    "v0":      {"color": STYLE["v0"]["color"],      "label": "v0 (no heuristic)",  "ls": ":"},
    "v1":      {"color": STYLE["v1"]["color"],      "label": "v1 (heuristic)",     "ls": "--"},
    "v2":      {"color": STYLE["v2"]["color"],      "label": "v2 (heuristic+TT)",  "ls": "-."},
    "current": {"color": STYLE["current"]["color"], "label": "current (TT+adapt)", "ls": "-"},
}

for agent_name, sub in timing_summary.groupby("agent"):
    if agent_name not in agent_style2:
        continue
    sub = sub.sort_values("piece_bin")
    s   = agent_style2[agent_name]
    ax.plot(sub["piece_bin"], sub["median"], marker="o", color=s["color"],
            linewidth=2, markersize=5, label=s["label"], linestyle=s["ls"])
    ax.fill_between(sub["piece_bin"], sub["q25"], sub["q75"],
                    alpha=0.12, color=s["color"])

ax.set_xlabel("Pieces already on board (game progress)")
ax.set_ylabel("Median decision time (ms)")
ax.set_title(f"Decision time by game stage — depth={REPORT_DEPTH}\n(IQR band, spec plays as P1 vs random)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig_time_by_move.pdf", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "fig_time_by_move.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_time_by_move")

### Pairwise Win Rate Heatmap: random vs v0 vs v1 vs v2 vs current

Runs every pair combination (N=20 games each, alternating starts).
The heatmap cell [i, j] shows agent-i's win rate against agent-j.
Diagonal = 0.5 (agent vs itself).  Lower triangle is the complement of upper triangle.

In [ ]:
HEATMAP_GAMES = 20
HEATMAP_AGENTS = ["random", "v0", "v1", "v2", "current"]

# heat_data[a][b] = a's win rate vs b
heat_data = {a: {b: 0.5 for b in HEATMAP_AGENTS} for a in HEATMAP_AGENTS}

for i, a_name in enumerate(HEATMAP_AGENTS):
    for b_name in HEATMAP_AGENTS[i+1:]:
        print(f"  {a_name} vs {b_name}...", flush=True)
        a_spec = POLICIES_REPORT[a_name]
        b_spec = POLICIES_REPORT[b_name]
        df = match(a_spec, b_spec, games=HEATMAP_GAMES, depth=REPORT_DEPTH)
        a_wr = (df["winner"] == a_name).mean()
        heat_data[a_name][b_name] = float(a_wr)
        heat_data[b_name][a_name] = float(1.0 - a_wr)

import numpy as np
heat_matrix = np.array([[heat_data[a][b] for b in HEATMAP_AGENTS] for a in HEATMAP_AGENTS])
print("\nHeatmap matrix (rows=agent, cols=opponent):")
print(pd.DataFrame(heat_matrix, index=HEATMAP_AGENTS, columns=HEATMAP_AGENTS).round(2))

In [ ]:
import matplotlib.colors as mcolors

agent_labels = [
    "random", "v0\n(no heuristic)", "v1\n(heuristic)",
    "v2\n(heuristic+TT)", "current\n(TT+adapt.)"
]

fig, ax = plt.subplots(figsize=(6.5, 5.5))
cmap = plt.cm.RdYlGn
im = ax.imshow(heat_matrix, cmap=cmap, vmin=0.0, vmax=1.0)

ax.set_xticks(range(len(HEATMAP_AGENTS)))
ax.set_yticks(range(len(HEATMAP_AGENTS)))
ax.set_xticklabels(agent_labels, fontsize=8)
ax.set_yticklabels(agent_labels, fontsize=8)
ax.set_xlabel("Opponent", fontsize=9)
ax.set_ylabel("Agent (row wins against column)", fontsize=9)
ax.set_title(f"Pairwise win rates (depth={REPORT_DEPTH}, N={HEATMAP_GAMES} games each)\n"
             "diagonal=0.5 by definition", fontsize=9)

for i in range(len(HEATMAP_AGENTS)):
    for j in range(len(HEATMAP_AGENTS)):
        v = heat_matrix[i, j]
        txt_color = "black" if 0.25 < v < 0.75 else "white"
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                fontsize=9, color=txt_color, fontweight="bold")

plt.colorbar(im, ax=ax, label="Win rate (row vs column)", fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig_heatmap.pdf", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "fig_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_heatmap")

### Iso-latency (Fair Depth) Comparison

Each agent receives the depth it can "afford" within a roughly comparable compute budget:
v0 → depth 8, v1 → depth 10 (~366 ms/move), v2 → depth 14 (~1057 ms/move), current → depth 14 (~1094 ms/move).
This removes the "TT makes depth cheap" distortion and asks: *at a fair budget, which agent wins?*

> **Note:** This cell takes ~5–8 minutes (v2/current at depth 14 take ~1094 ms/move).

In [ ]:
ISO_DEPTHS  = {"v0": 8, "v1": 10, "v2": 14, "current": 14}
N_ISO_GAMES = 5  # ~5-8 min total due to v2/current@14 (~1094 ms/move)

def play_game_mixed(a_name, b_name, a_depth, b_depth, a_is_p1):
    """Play one game where each agent uses its own assigned depth."""
    a_spec = POLICIES_REPORT[a_name]
    b_spec = POLICIES_REPORT[b_name]
    if a_is_p1:
        p1_pol, p2_pol = make_policy(a_spec, a_depth), make_policy(b_spec, b_depth)
        p1_name, p2_name = a_name, b_name
    else:
        p1_pol, p2_pol = make_policy(b_spec, b_depth), make_policy(a_spec, a_depth)
        p1_name, p2_name = b_name, a_name
    state = ConnectState()
    while not state.is_final():
        pol = p1_pol if state.player == -1 else p2_pol
        state = state.transition(int(pol.act(state.board)))
    wc = state.get_winner()
    if wc == -1:  return p1_name
    if wc == +1:  return p2_name
    return "draw"

ISO_AGENTS = ["v0", "v1", "v2", "current"]
iso_data   = {a: {b: 0.5 for b in ISO_AGENTS} for a in ISO_AGENTS}

for i, a in enumerate(ISO_AGENTS):
    for b in ISO_AGENTS[i+1:]:
        print(f"  {a}(d={ISO_DEPTHS[a]}) vs {b}(d={ISO_DEPTHS[b]})...", flush=True)
        a_wins = sum(
            1 for g in range(N_ISO_GAMES)
            if play_game_mixed(a, b, ISO_DEPTHS[a], ISO_DEPTHS[b], a_is_p1=(g % 2 == 0)) == a
        )
        wr = a_wins / N_ISO_GAMES
        iso_data[a][b] = wr
        iso_data[b][a] = 1.0 - wr

iso_matrix = np.array([[iso_data[a][b] for b in ISO_AGENTS] for a in ISO_AGENTS])
print(pd.DataFrame(iso_matrix, index=ISO_AGENTS, columns=ISO_AGENTS).round(2))

In [ ]:
iso_labels = [f"{a}\n(d={ISO_DEPTHS[a]})" for a in ISO_AGENTS]

fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = ax.imshow(iso_matrix, cmap=plt.cm.RdYlGn, vmin=0.0, vmax=1.0)

ax.set_xticks(range(len(ISO_AGENTS)))
ax.set_yticks(range(len(ISO_AGENTS)))
ax.set_xticklabels(iso_labels, fontsize=9)
ax.set_yticklabels(iso_labels, fontsize=9)
ax.set_xlabel("Opponent", fontsize=9)
ax.set_ylabel("Agent (row wins against column)", fontsize=9)
ax.set_title(f"Iso-latency win rates (N={N_ISO_GAMES} games, each agent at its compute-budget depth)\n"
             "diagonal = 0.5 by definition", fontsize=9)

for i in range(len(ISO_AGENTS)):
    for j in range(len(ISO_AGENTS)):
        v = iso_matrix[i, j]
        txt_color = "black" if 0.25 < v < 0.75 else "white"
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                fontsize=10, color=txt_color, fontweight="bold")

plt.colorbar(im, ax=ax, label="Win rate (row vs column)", fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig_iso_heatmap.pdf", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "fig_iso_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_iso_heatmap")

### Adaptive Deepening in Action: Single-Game Depth Trace

Shows the exact search depth used by `current` on each of its moves in one live game against random
with a 10-second timeout.  Starting at depth 14 (~1094 ms), after the first move the condition
`elapsed × 7.5 < timeout × 0.9` is satisfied (1094 × 7.5 = 8205 ms < 9000 ms),
so `search_depth` steps up to 16.  At depth 16 (~7341 ms) the same condition fails
(7341 × 7.5 = 55 057 ms ≫ 9000 ms), so the agent stabilises at depth 16 for all remaining moves.

> **Note:** This cell takes ~40–70 s (depth-16 moves take ~7.3 s each).

In [ ]:
ADAPT_TIMEOUT_S = 10.0   # seconds — matches elapsed = time.time()-start (also seconds)
MAX_ADAPT_MOVES = 20     # cap total moves to keep runtime ~60 s

pol_adapt = make_policy_with_timeout(POLICIES_REPORT["current"], timeout=ADAPT_TIMEOUT_S)
pol_rnd   = make_policy(POLICIES_REPORT["random"])

state = ConnectState()
adapt_records = []
move_num = 0

while not state.is_final() and move_num < MAX_ADAPT_MOVES:
    move_num += 1
    if state.player == -1:                      # current plays as P1 (Red)
        depth_before = getattr(pol_adapt, "search_depth", None)
        t0     = time.perf_counter()
        action = int(pol_adapt.act(state.board))
        ms     = (time.perf_counter() - t0) * 1000
        depth_after = getattr(pol_adapt, "search_depth", None)
        adapt_records.append({
            "current_move": len(adapt_records) + 1,
            "game_move":    move_num,
            "ms":           ms,
            "depth_used":   depth_before,
            "depth_next":   depth_after,
        })
        print(f"  Move {len(adapt_records):2d}: depth_used={depth_before} → next={depth_after}, "
              f"time={ms:7.0f} ms", flush=True)
    else:
        action = int(pol_rnd.act(state.board))
    state = state.transition(action)

adapt_df = pd.DataFrame(adapt_records)
wc = state.get_winner()
print(f"\nGame ended at total move {move_num}. "
      f"Winner: {'current (Red)' if wc==-1 else 'random (Yellow)' if wc==1 else 'draw'}")
adapt_df

In [ ]:
if len(adapt_df) == 0:
    print("No adaptive data — game ended before current moved")
else:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5.2), sharex=True,
                                    gridspec_kw={"height_ratios": [2, 1]})

    x     = adapt_df["current_move"]
    color = STYLE["current"]["color"]

    # threshold in ms: elapsed(s) * 7.5 < timeout * 0.9  →  elapsed < timeout*0.9/7.5
    threshold_ms = ADAPT_TIMEOUT_S * 0.9 / 7.5 * 1000   # 10 * 0.9 / 7.5 * 1000 = 1200 ms

    # Top: decision time per move
    ax1.step(x, adapt_df["ms"], where="post", color=color, linewidth=2.5, zorder=3)
    ax1.scatter(x, adapt_df["ms"], color=color, s=65, zorder=5)
    ax1.axhline(threshold_ms, color="crimson", linestyle=":", linewidth=1.5,
                label=f"Budget threshold ({threshold_ms:.0f} ms)\n"
                      f"= timeout({ADAPT_TIMEOUT_S:.0f} s)×0.9÷7.5×1000")
    ax1.set_ylabel("Decision time (ms)")
    ax1.set_yscale("log")
    ax1.set_title(f"Adaptive deepening trace — current vs random (timeout={ADAPT_TIMEOUT_S:.0f} s)")
    ax1.legend(fontsize=8)

    # Bottom: depth used per move
    ax2.step(x, adapt_df["depth_used"], where="post", color=color, linewidth=2.5, zorder=3)
    ax2.scatter(x, adapt_df["depth_used"], color=color, s=65, zorder=5)
    ax2.set_ylabel("Search depth")
    ax2.set_xlabel("Move index (current's moves only)")
    depth_vals = sorted(adapt_df["depth_used"].unique())
    ax2.set_yticks(depth_vals)
    ax2.set_xticks(x)

    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "fig_adaptive_trace.pdf", bbox_inches="tight")
    fig.savefig(FIGURE_DIR / "fig_adaptive_trace.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved fig_adaptive_trace")

### Current vs Current: Per-Move Decision Time for Both Players

Two independent `current` instances (each with its own fresh transposition table) play head-to-head
at a fixed depth.  Since P1 and P2 build separate caches, their TT warmup trajectories are
independent — but should converge to similar patterns (same algorithm, symmetric positions).

In [ ]:
N_CVC    = 5    # games to aggregate
CVC_DEPTH = 10  # fixed depth: ~17 ms/move, ~4 s for 5 games

cvc_records = []
spec_c = POLICIES_REPORT["current"]
for g in range(N_CVC):
    p1 = make_policy(spec_c, CVC_DEPTH)
    p2 = make_policy(spec_c, CVC_DEPTH)
    p1_m = p2_m = 0
    state = ConnectState()
    while not state.is_final():
        n_pieces = int(np.sum(state.board != 0))
        if state.player == -1:
            p1_m += 1
            t0     = time.perf_counter()
            action = int(p1.act(state.board))
            cvc_records.append({"game": g+1, "player": "P1 (Red)",
                                 "move": p1_m, "pieces": n_pieces,
                                 "ms": (time.perf_counter() - t0) * 1000})
        else:
            p2_m += 1
            t0     = time.perf_counter()
            action = int(p2.act(state.board))
            cvc_records.append({"game": g+1, "player": "P2 (Yellow)",
                                 "move": p2_m, "pieces": n_pieces,
                                 "ms": (time.perf_counter() - t0) * 1000})
        state = state.transition(action)
    print(f"  Game {g+1}: winner color {state.get_winner()}", flush=True)

cvc_df = pd.DataFrame(cvc_records)
cvc_df["piece_bin"] = (cvc_df["pieces"] // 4) * 4
cvc_summary = (cvc_df.groupby(["player", "piece_bin"])["ms"]
               .agg(median="median",
                    q25=lambda x: x.quantile(0.25),
                    q75=lambda x: x.quantile(0.75))
               .reset_index())

fig, ax = plt.subplots(figsize=(7, 4))
pal = {"P1 (Red)": "#dc2626", "P2 (Yellow)": "#d97706"}
for player, sub in cvc_summary.groupby("player"):
    sub = sub.sort_values("piece_bin")
    c   = pal[player]
    ax.plot(sub["piece_bin"], sub["median"], marker="o", color=c,
            linewidth=2, markersize=6, label=player)
    ax.fill_between(sub["piece_bin"], sub["q25"], sub["q75"], alpha=0.15, color=c)

ax.set_xlabel("Pieces on board (game progress)")
ax.set_ylabel("Median decision time (ms)")
ax.set_title(f"Current vs Current (depth={CVC_DEPTH}, N={N_CVC} games)\n"
             "Both TTs start cold — warmup reduces latency as game progresses")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig_current_vs_current.pdf", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "fig_current_vs_current.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_current_vs_current")

### Column Preference: Where Does Each Agent Play?

Reveals whether agents develop a strategic column preference.  The center column (3) maximises
the number of possible 4-in-a-row lines, so any agent that understands the game should
prefer it — especially on the opening move from an empty board.

In [ ]:
N_COL_GAMES = 15  # fast at depth 8 (~2.6 ms/move)

def collect_col_distribution(agent_name, n_games=N_COL_GAMES):
    """Collect all columns chosen by agent (P1) vs random (P2) at depth 8."""
    spec     = POLICIES_REPORT[agent_name]
    rnd_spec = POLICIES_REPORT["random"]
    cols = []
    for _ in range(n_games):
        pol   = make_policy(spec, REPORT_DEPTH)
        rnd_p = make_policy(rnd_spec)
        state = ConnectState()
        while not state.is_final():
            if state.player == -1:
                col = int(pol.act(state.board))
                cols.append(col)
                state = state.transition(col)
            else:
                state = state.transition(int(rnd_p.act(state.board)))
    return cols

col_data = {}
for agent_name in ["v0", "v1", "v2", "current"]:
    print(f"  {agent_name}...", flush=True)
    col_data[agent_name] = collect_col_distribution(agent_name)

# First move on the empty board
first_moves = {a: int(make_policy(POLICIES_REPORT[a], REPORT_DEPTH).act(ConnectState().board))
               for a in ["v0", "v1", "v2", "current"]}
print("Opening move (empty board):", first_moves)

fig, axes = plt.subplots(1, 4, figsize=(11, 3.4), sharey=True)
for ax, agent_name in zip(axes, ["v0", "v1", "v2", "current"]):
    cols   = col_data[agent_name]
    counts = np.bincount(cols, minlength=7)
    probs  = counts / counts.sum()
    first  = first_moves[agent_name]
    bcolors = [("#dc2626" if c == first else STYLE[agent_name]["color"]) for c in range(7)]
    ax.bar(range(7), probs, color=bcolors, alpha=0.85, edgecolor="white", linewidth=0.5)
    ax.axvline(first, color="crimson", linestyle="--", linewidth=1.5, alpha=0.8)
    ax.set_title(f"{agent_name}\n(depth={REPORT_DEPTH})", fontsize=9)
    ax.set_xlabel("Column")
    ax.set_xticks(range(7))
    if ax == axes[0]:
        ax.set_ylabel("Fraction of moves")
    ax.text(first, probs.max() * 0.95, f"↑{first}", ha="center", va="top",
            color="crimson", fontsize=8, fontweight="bold")

fig.suptitle(f"Column preference as P1 vs random (N={N_COL_GAMES} games, depth={REPORT_DEPTH})\n"
             "Red bar/dashed = opening move on empty board", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig_column_preference.pdf", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "fig_column_preference.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_column_preference")

### Game Length Distribution: How Quickly Does Each Agent Win?

Stronger agents win faster because they steer toward decisive positions rather than drifting.
Random games tend to fill the board; heuristic agents exploit early patterns.

In [ ]:
N_LEN_GAMES = 20

def game_lengths_vs_random(agent_name, depth, n=N_LEN_GAMES):
    spec = POLICIES_REPORT[agent_name]
    rnd  = POLICIES_REPORT["random"]
    lengths = []
    for i in range(n):
        if i % 2 == 0:
            p1_pol, p2_pol = make_policy(spec, depth), make_policy(rnd)
        else:
            p1_pol, p2_pol = make_policy(rnd), make_policy(spec, depth)
        state = ConnectState()
        moves = 0
        while not state.is_final() and moves < 42:
            pol    = p1_pol if state.player == -1 else p2_pol
            state  = state.transition(int(pol.act(state.board)))
            moves += 1
        lengths.append(moves)
    return lengths

len_data = {}
for agent_name in ["random", "v0", "v1", "v2", "current"]:
    print(f"  {agent_name}...", flush=True)
    depth = REPORT_DEPTH if agent_name != "random" else None
    if agent_name == "random":
        # random vs random
        lengths = []
        for _ in range(N_LEN_GAMES):
            p1, p2 = make_policy(POLICIES_REPORT["random"]), make_policy(POLICIES_REPORT["random"])
            state  = ConnectState()
            moves  = 0
            while not state.is_final() and moves < 42:
                state = state.transition(int((p1 if state.player==-1 else p2).act(state.board)))
                moves += 1
            lengths.append(moves)
        len_data[agent_name] = lengths
    else:
        len_data[agent_name] = game_lengths_vs_random(agent_name, REPORT_DEPTH)

fig, ax = plt.subplots(figsize=(8, 4))
agent_order = ["random", "v0", "v1", "v2", "current"]
style_ext   = {**STYLE, "random": {"color": "#9ca3af", "label": "random"}}
all_lengths  = [len_data[a] for a in agent_order]
colors_box   = [style_ext[a]["color"] for a in agent_order]
labels_box   = [f"{a}\n(vs random)" if a != "random" else "random\nvs random"
                for a in agent_order]

bp = ax.boxplot(all_lengths, patch_artist=True, widths=0.5,
                medianprops={"color": "black", "linewidth": 2})
for patch, c in zip(bp["boxes"], colors_box):
    patch.set_facecolor(c)
    patch.set_alpha(0.75)

ax.set_xticklabels(labels_box, fontsize=8)
ax.set_ylabel("Game length (total moves)")
ax.set_title(f"Game length distribution vs random (N={N_LEN_GAMES}, depth={REPORT_DEPTH})\n"
             "Stronger agents win faster — fewer moves needed to find decisive lines")
ax.set_ylim(0, 45)
ax.axhline(42, color="gray", linestyle="--", linewidth=1, alpha=0.5, label="Board full")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig_game_length.pdf", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "fig_game_length.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_game_length")

## Same-Depth 8 Notes

For a strict same-depth comparison, instantiate each policy with `depth=8` using `make_policy(spec, depth=8)` or call the helpers with `depth=8`. The remaining `comptime DEPTH` constants are now just defaults/reference values for these benchmark cells.